# Molecular Devices FilterMax F5

```{device-card} molecular-devices-filtermax-f5
```

| Property | Value |
|---|---|
| PLR class | `FilterMaxF5` |
| Connection | RS-232 serial, 38,400 baud, 7E1, no flow control |
| Current verified modes | Absorbance and luminescence |
| Current verified controls | Tray, filter slides, temperature, shaking, cancellation |

> **Validation status:** Direct PLR-to-F5 hardware validation succeeded. Full-plate absorbance at 450 nm agreed with the corresponding SoftMax Pro results; additional validated operations are recorded in the [protocol coverage](protocol-coverage.md). Fluorescence, FP, and TRF commands remain unavailable until they can be captured with a compatible excitation slide; the driver does not infer them.

## How it communicates

The F5 uses an ASTM-like framed serial protocol, not the SpectraMax `!COMMAND` protocol. PLR handles framing, checksums, acknowledgements, response chunking, device errors, and cancellation internally. Raw commands are intentionally not public.

## Physical setup

1. Connect the FilterMax F5 through its RS-232 cable or a USB-to-serial adapter.
2. Close SoftMax Pro and release the serial port. Only one program may own the port.
3. Export the filter-slide definitions from SoftMax Pro as XML.
4. Install the plate and filter slides appropriate for the requested measurement.
5. Use a stable device path on Linux, such as `/dev/serial/by-id/...`.

Excitation and emission slides occupy separate bays and are not interchangeable.

## Create the reader

Load the SoftMax filter-slide catalog. Measurement methods require it so wavelengths can be resolved to an exact installed slide and slot.

In [ ]:
from pathlib import Path

from pylabrobot.molecular_devices.filtermax import FilterMaxF5, FilterSlideCatalog

catalog_path = Path("~/Documents/WIN7Shared/filtermax_slides.xml").expanduser()
catalog = FilterSlideCatalog.from_softmax_xml(catalog_path)
reader = FilterMaxF5(
    name="filtermax-f5",
    port="/dev/serial/by-id/replace-with-your-adapter",
    filter_slide_catalog=catalog,
)

## Connect

`setup()` opens the serial port, reads the instrument identity, and refuses a device that is not an F5.

In [ ]:
await reader.setup()

## Inspect the instrument

Read identity, status, installed slide IDs, and temperature without changing instrument state.

In [ ]:
info = await reader.get_instrument_info()
status = await reader.get_status()
slides = await reader.get_filter_slides()
temperature = await reader.get_temperature()

info, status, slides, temperature

## Plate tray

Move the tray out before loading or removing a plate.

In [ ]:
await reader.move_plate_tray_out()

After the plate is loaded and the tray path is clear, move it back in.

In [ ]:
await reader.move_plate_tray_in()

## Temperature control

Set a temperature in degrees Celsius, then query the current chamber temperature.

In [ ]:
await reader.set_temperature(25.0)
temperature = await reader.get_temperature()

Deactivate temperature control when it is no longer needed.

In [ ]:
await reader.deactivate_temperature_control()

## Shaking

Shaking is a finite-duration operation. Patterns are `"linear"` or `"orbital"`; speeds are `"low"`, `"medium"`, or `"high"`.

In [ ]:
await reader.start_shaking(pattern="linear", speed="medium", duration=5)

## Plate geometry

The captured Costar 96-well clear plate geometry is available as a convenience. For another plate, construct `PlateGeometry` using its measured dimensions in millimetres.

In [ ]:
from pylabrobot.molecular_devices.filtermax import PlateGeometry

plate = PlateGeometry.costar_96_clear_landscape()

## Absorbance

Run an endpoint read at 450 nm. Results are plate-shaped optical-density values; unselected wells are `None`. The tray ejects when the read finishes.

In [ ]:
absorbance = await reader.read_absorbance(plate, wavelength=450)

Select individual wells for a partial-plate read.

In [ ]:
partial = await reader.read_absorbance(
    plate,
    wavelength=450,
    wells=["C4", "C5", "C6", "D4", "D5", "D6"],
)

Use a supported reference filter when reference-subtracted absorbance is required.

In [ ]:
reference_subtracted = await reader.read_absorbance(
    plate,
    wavelength=450,
    reference_wavelength=620,
    wells=["C4", "C5", "C6", "D4", "D5", "D6"],
)

## Kinetic reads

Kinetic timing uses seconds. The result list contains one plate-shaped result per timepoint.

In [ ]:
from pylabrobot.molecular_devices.filtermax import KineticTiming

kinetic = await reader.read_absorbance(
    plate,
    wavelength=450,
    wells=["C4"],
    kinetic=KineticTiming(interval=15, reads=3),
)

## Well scans

The captured F5 software exposes horizontal and circular fill scans. Scan results preserve every `(x, y, value)` point for each selected well.

In [ ]:
from pylabrobot.molecular_devices.filtermax import WellScanSettings

scan = await reader.read_absorbance(
    plate,
    wavelength=450,
    wells=["C4"],
    well_scan=WellScanSettings(pattern="fill", density=5),
)

## Luminescence

Read the open luminescence position on the installed emission slide. Values are raw RLU. Set `channels=2` for the captured dual-luminescence operation.

In [ ]:
luminescence = await reader.read_luminescence(
    plate,
    wells=["C4"],
    integration=0.4,
    read_height=1.0,
)

## Cancel a read

`cancel_read()` is state-aware: it does nothing while idle and cooperatively stops an active read. A cancelled read raises `FilterMaxReadCancelled` in the read task and returns the reader to an idle, tray-ejected state.

In [ ]:
import asyncio

from pylabrobot.molecular_devices.filtermax import FilterMaxReadCancelled

read_task = asyncio.create_task(
    reader.read_absorbance(
        plate,
        wavelength=450,
        wells=["C4"],
        kinetic=KineticTiming(interval=30, reads=10),
    )
)
await asyncio.sleep(2)
await reader.cancel_read()
try:
    await read_task
except FilterMaxReadCancelled:
    pass

## Device errors

Inspect the exact firmware errors observed during this PLR session, then clear the local session log when handled.

In [ ]:
errors = await reader.get_error_log()
await reader.clear_error_log()

## Disconnect

Always release the serial port when finished.

In [ ]:
await reader.stop()